## **GOLD DATA EXTRACTION AND FEATURE ENGINEERING** 
#### **Description:**
This cell extracts gold price data from Yahoo Finance and performs controlled feature engineering. It ensures consistent structure, handles missing values, and avoids premature rounding to preserve analytical accuracy for downstream layers.

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# -------------------------------
# Define dynamic date range
# -------------------------------
start_date = "2025-01-01"
end_date = (datetime.today() + timedelta(days=1)).strftime('%Y-%m-%d')

# -------------------------------
# Extract data
# -------------------------------
gold = yf.download(
    "GC=F",
    start=start_date,
    end=end_date,
    interval="1d"
)

# -------------------------------
# Reset structure
# -------------------------------
gold = gold.reset_index()

# -------------------------------
# Select relevant fields
# -------------------------------
gold = gold[['Date', 'Close']]
gold.columns = ['process_date', 'Gold_Ounce_USD']

# -------------------------------
# Handle missing values
# -------------------------------
gold = gold.dropna(subset=['Gold_Ounce_USD'])

# -------------------------------
# Feature engineering
# -------------------------------
gold["Gold_Gram_USD"] = gold["Gold_Ounce_USD"] / 31.1035
gold["Gold_24K"] = gold["Gold_Gram_USD"]
gold["Gold_21K"] = gold["Gold_Gram_USD"] * 0.875
gold["Gold_18K"] = gold["Gold_Gram_USD"] * 0.75

# -------------------------------
# Ensure correct data types
# -------------------------------
gold["process_date"] = pd.to_datetime(gold["process_date"])

# Keep high precision (no rounding here)

## **EXPORT TO BRONZE LAYER FILE**
#### **Description:**
This cell exports the cleaned dataset into a structured CSV file that serves as the ingestion source for the Bronze layer in the data warehouse pipeline. It ensures consistent encoding and stable schema.

In [ ]:
gold.to_csv(
    "gold_prices_2025_to_today.csv",
    index=False,
    encoding="utf-8"
)

## **DATA QUALITY VALIDATION**
#### **Description:**
This cell reloads the exported dataset to verify structural integrity, confirm schema correctness, and ensure no corruption occurred during export.

In [ ]:
df = pd.read_csv("gold_prices_2025_to_today.csv")

# Basic validation checks
print(df.info())
print(df.head())

## **DATE STANDARDIZATION (IMPROVED SAFE VERSION)**
#### **Description:**
This cell standardizes the date format and ensures compatibility with SQL ingestion. It safely handles invalid values and guarantees a consistent YYYY-MM-DD format for downstream database loading.

In [ ]:
df["process_date"] = pd.to_datetime(df["process_date"], errors="coerce")

# Remove invalid dates
df = df.dropna(subset=["process_date"])

# Standard SQL-compatible format
df["process_date"] = df["process_date"].dt.strftime("%Y-%m-%d")